In [ ]:
# setup
import pandas as pd
import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch


pd.options.display.max_colwidth = 100
pd.options.display.max_rows = 10
pd.options.display.max_columns = 30

In [ ]:
# data load 

# paths
path_onedrive = "/Users/laurenwilner/Library/CloudStorage/OneDrive-SharedLibraries-UW/casey_cohort - Documents/studies/la_wf_pm_evac_its/"
path_repo = "~/Desktop/Desktop/epidemiology_PhD/00_repos/la-wf"
path_data = "/Users/laurenwilner/Desktop/Desktop/epidemiology_PhD/01_data"

data_dir = path_onedrive + '01_data/02_processed/01_exposure/'
raw_dir = path_onedrive + '01_data/01_raw/'

# read in exposure data 
pm_exp = pd.read_csv(data_dir + 'exposed_cts_pm.csv')
evac_exp = pd.read_csv(data_dir + 'exposed_cts_evac.csv')

# census tracts: just need to send data to kpsc with ct geoid
cts = gpd.read_file(raw_dir + 'tl_2010_06_tract10.shp')
cts = cts[['GEOID10', 'geometry']].rename(columns={'GEOID10': 'geoid'})

# CA state boundary for trimming coastal CT
states = gpd.read_file(raw_dir + 'cb_2018_us_state_500k.shp')
ca_state = states[states['STUSPS'] == 'CA'].reset_index(drop=True)
ca_state = ca_state[['geometry']]
ca_state = ca_state.to_crs(cts.crs)

# get rid of tracts in the ocean
cts = gpd.overlay(cts, ca_state, how='intersection') # intersect with CA state boundary

# fire boundaries
fires = gpd.read_file(raw_dir + 'data_2025_01_17.geojson').to_crs(epsg=2229)
fires["poly_DateCurrent"] = fires["poly_DateCurrent"].dt.tz_convert('US/Pacific')
fires = fires[fires['poly_DateCurrent'] > '2025-01-06']
fires["poly_DateCurrent"] = fires["poly_DateCurrent"].dt.date
fires = fires[['geometry']]
fires_union = fires.dissolve()  # dissolve to one multipolygon
fires_union = fires_union.to_crs(cts.crs)  # convert to same CRS as evac data


In [ ]:
# combine exposure data
# data dictionary: 
    # geoid -- census tract id
    # exposure_category
        # evac (anywhere that is evacuated goes here and smoke is ignored)
        # high smoke -- no evac
        # mid smoke -- no evac 
        # no smoke -- no evac 

# merge pm and evacuation exposure data
exp = pm_exp.merge(evac_exp, on='geoid', how='left')
exp['exposed_evac'] = exp['exposed_evac'].fillna(0).astype(int)

# define exposure categories
conditions = [
    exp['exposed_evac'] == 1,
    (exp['exposed_evac'] == 0) & (exp['exposed_pm'] == 'high'),
    (exp['exposed_evac'] == 0) & (exp['exposed_pm'] == 'mid'),
    (exp['exposed_evac'] == 0) & (exp['exposed_pm'] == 'none')
]
categories = ['evac', 'high smoke, no evac', 'mid smoke, no evac', 'no smoke, no evac']

# assign exposure category to each row based on conditions
exp['exposure_category'] = np.select(conditions, categories, default='NA')

exp

In [ ]:
# write out  exposure data
exp.to_csv(data_dir + 'kpsc_pm_evac_exposure.csv', index=False)

In [ ]:
# diagnostic plot!

### create plot data
# pad the geoid in exp with leading zero and make it a string
exp['geoid'] = exp['geoid'].astype(str).str.zfill(11)  # pad to 11 digits
# make ct geoid a string
cts['geoid'] = cts['geoid'].astype(str)
# merge
merged_data = cts.merge(exp, on='geoid', how='inner')  # Use inner join to only keep tracts with data

### make plot
fig, ax = plt.subplots(1, 1, figsize=(14, 10))

# color map
color_map = {
    'no smoke, no evac': '#C0E6E0',      
    'mid smoke, no evac': '#73C1B9',     
    'high smoke, no evac': '#2E8B8B',    
    'evac': '#0C5985'                    
}

# plot each exposure category
for category, color in color_map.items():
    subset = merged_data[merged_data['exposure_category'] == category]
    if not subset.empty:
        subset.plot(ax=ax, color=color, label=category)

# plot fire boundary
fires_union.plot(ax=ax, facecolor='#9A3334B3', edgecolor='#9A3334', linewidth=2, label='Fire boundaries')


# remove axis
ax.set_xticks([])
ax.set_yticks([])

# remove border 
for spine in ax.spines.values():
    spine.set_visible(False)

# add legend 
legend_elements = [
    Patch(facecolor='#C0E6E0', label='No smoke, no evacuation'),
    Patch(facecolor='#73C1B9', label='Mid smoke, no evacuation'),
    Patch(facecolor='#2E8B8B', label='High smoke, no evacuation'),
    Patch(facecolor='#0C5985', label='Evacuation'),
    Patch(facecolor='none', edgecolor='#9A3334', linewidth=2, label='Fire boundaries')
]
plt.legend(handles=legend_elements, bbox_to_anchor=(1.05, 1), loc='upper left', fontsize = 16, frameon=True)

# add summary stats to the plot
summary_stats = merged_data['exposure_category'].value_counts()
stats_text = "Summary of exposure categories:\n"
for category, count in summary_stats.items():
    stats_text += f"{category}: {count}\n"
stats_text += f"\nTotal tracts: {len(merged_data)}"

plt.text(-0.1, 0, stats_text, transform=ax.transAxes, fontsize=16, 
         verticalalignment='bottom', bbox=dict(boxstyle="round,pad=0.3", facecolor="none", alpha=0.8))


plt.tight_layout()
plt.subplots_adjust(right=0.8)

plt.show()

In [ ]:
# diagnostic plot!

### create plot data
# pad the geoid in exp with leading zero and make it a string
exp['geoid'] = exp['geoid'].astype(str).str.zfill(11)  # pad to 11 digits
# make ct geoid a string
cts['geoid'] = cts['geoid'].astype(str)
# merge
merged_data = cts.merge(exp, on='geoid', how='inner')  # Use inner join to only keep tracts with data

### make plot
fig, ax = plt.subplots(1, 1, figsize=(14, 10))

# color map
color_map = {
    'no smoke, no evac': '#C0E6E0',      
    'mid smoke, no evac': '#73C1B9',     
    'high smoke, no evac': '#2E8B8B',    
    'evac': '#0C5985'                    
}

# plot each exposure category
for category, color in color_map.items():
    subset = merged_data[merged_data['exposure_category'] == category]
    if not subset.empty:
        subset.plot(ax=ax, color=color, label=category)

# plot fire boundary
fires_union.plot(ax=ax, facecolor='#9A3334B3', edgecolor='#9A3334', linewidth=2, label='Fire boundaries')

# Calculate bounding box of the merged data (non-"none" exposure areas)
bounds = merged_data[merged_data['exposure_category'] != 'no smoke, no evac'].total_bounds  # returns [minx, miny, maxx, maxy]
minx, miny, maxx, maxy = bounds

# Add some padding (optional - adjust the padding factor as needed)
padding_factor = 0.05  # 5% padding
x_padding = (maxx - minx) * padding_factor
y_padding = (maxy - miny) * padding_factor

# Set axis limits to zoom in on the data
ax.set_xlim(minx - x_padding, maxx + x_padding)
ax.set_ylim(miny - y_padding, maxy + y_padding)

# remove axis
ax.set_xticks([])
ax.set_yticks([])

# remove border 
for spine in ax.spines.values():
    spine.set_visible(False)

# add legend 
legend_elements = [
    Patch(facecolor='#C0E6E0', label='No smoke, no evacuation'),
    Patch(facecolor='#73C1B9', label='Mid smoke, no evacuation'),
    Patch(facecolor='#2E8B8B', label='High smoke, no evacuation'),
    Patch(facecolor='#0C5985', label='Evacuation'),
    Patch(facecolor='none', edgecolor='#9A3334', linewidth=2, label='Fire boundaries')
]
plt.legend(handles=legend_elements, bbox_to_anchor=(1.05, 1), loc='upper left', fontsize = 16, frameon=True)

# add summary stats to the plot
summary_stats = merged_data['exposure_category'].value_counts()
stats_text = "Summary of exposure categories:\n"
for category, count in summary_stats.items():
    stats_text += f"{category}: {count}\n"
stats_text += f"\nTotal tracts: {len(merged_data)}"

plt.text(-0.1, 0, stats_text, transform=ax.transAxes, fontsize=16, 
         verticalalignment='bottom', bbox=dict(boxstyle="round,pad=0.3", facecolor="none", alpha=0.8))

plt.tight_layout()
plt.subplots_adjust(right=0.8)

plt.show()